In [1]:
# Extended Tutorial: Introduction to LSTM for Time Series Forecasting

# Objectives:
# - Understand the fundamentals of LSTM for sequential data.
# - Preprocess and prepare time series data for LSTM input.
# - Build, train, and evaluate LSTM models for time series forecasting.
# - Apply the concepts using an actual dataset (e.g., stock prices dataset).


# Content Overview:
# 1. Introduction to LSTM for Time Series Data
# 2. Preprocessing Time Series Data
# 3. Building a Basic LSTM Model for Time Series
# 4. Training and Evaluating the LSTM Model
# 5. Exercise: Time Series Forecasting Using a Real Dataset (Stock Prices)

In [ ]:
# Importing necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# ====================================
# 1. Introduction to LSTM for Time Series Data
# ====================================
# LSTMs (Long Short-Term Memory networks) are a type of Recurrent Neural Network (RNN) 
#designed to model and predict sequential data, such as time series.

# ====================================
# 2. Preprocessing Time Series Data
# ====================================
# For this tutorial, we will create a synthetic sine wave dataset to demonstrate 
# time series prediction.
# Later, we will work with a real-world dataset.

# Generate synthetic sine wave data
time_steps = 1000
x = np.linspace(0, 100, time_steps)
y = np.sin(x)

plt.plot(x, y)
plt.title('Synthetic Sine Wave')
plt.xlabel('Time')
plt.ylabel('Value')
plt.show()

In [ ]:
# Prepare data for LSTM (sliding window approach)
def create_dataset(data, look_back=1):
    X, Y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i:(i + look_back)])
        Y.append(data[i + look_back])
    return np.array(X), np.array(Y)

# Normalize the data
scaler = MinMaxScaler(feature_range=(0, 1))
y = scaler.fit_transform(y.reshape(-1, 1)).flatten()

# Create the dataset with look_back=10
look_back = 10
X, y = create_dataset(y, look_back)
X = X.reshape(X.shape[0], X.shape[1], 1)  # Reshape for LSTM input

# Split into training and testing sets
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]


In [ ]:
# ====================================
# 3. Building a Basic LSTM Model for Time Series
# ====================================

# Define the LSTM model using the Sequential API
model = Sequential()

# Add the first LSTM layer with 50 units (neurons)
# 'input_shape' specifies the shape of the input data: (look_back, 1), 
#where 'look_back' is the number of previous time steps used for prediction
# 'return_sequences=True' ensures that the output is a sequence, 
#which is required for stacking another LSTM layer

model.add(LSTM(50, input_shape=(look_back, 1), return_sequences=True))

# Add a Dropout layer with a rate of 0.2 (20%) for regularization, 
#helping to reduce overfitting by randomly setting 20% of the input 
#units to 0 at each update during training

model.add(Dropout(0.2))

# Add a second LSTM layer with 50 units; since 'return_sequences' is not set, 
#this layer outputs the final hidden state (not a sequence)

model.add(LSTM(50))

# Add a Dense (fully connected) layer with 1 unit; 
#this layer is used for outputting the predicted value 
# (e.g., next value in a time series)

model.add(Dense(1))

# Compile the model with the 'adam' optimizer and 'mean_squared_error' as the loss function
# 'adam' is an adaptive learning rate optimizer commonly used for deep learning models
# 'mean_squared_error' is used as the loss function, suitable for 
#regression tasks where we aim to minimize the squared difference 
# between actual and predicted values

model.compile(optimizer='adam', loss='mean_squared_error')

# Display the summary of the model, showing the layers, output shapes, 
# and the total number of parameters

model.summary()


In [ ]:
# ====================================
# 4. Training and Evaluating the LSTM Model
# ====================================

# Train the LSTM model using the training data
# 'epochs=20' specifies the number of times the entire training dataset 
# is passed through the model
# 'batch_size=16' indicates the number of samples processed 
# before the model's internal parameters are updated
# 'validation_data' specifies the data on which to evaluate the 
# loss and any model metrics at the end of each epoch
# 'verbose=1' enables progress output during training
history = model.fit(X_train, y_train, epochs=20, batch_size=16, validation_data=(X_test, y_test), verbose=1)

# Plot the training history to visualize the loss over epochs
# This helps to observe the model's learning progress and potential 
# overfitting or underfitting
plt.plot(history.history['loss'], label='Train Loss')  # Plot the training loss
plt.plot(history.history['val_loss'], label='Validation Loss')  # Plot the validation loss
plt.title('Model Loss During Training')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()  # Add a legend to distinguish between training and validation loss
plt.show()

# Make predictions on the test data
y_pred = model.predict(X_test)  # Predict values using the trained model
# Inverse transform the predictions to get them back to the original scale
y_pred = scaler.inverse_transform(y_pred)
# Inverse transform the true test values for comparison
y_test = scaler.inverse_transform(y_test.reshape(-1, 1))

# Evaluate the model's performance using the Mean Squared Error (MSE) metric
# MSE measures the average squared difference between actual and predicted values
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error on Test Set: {mse}')

# Plot the true values and predicted values to visualize the model's performance
plt.plot(y_test, label='True Value')  # Plot true values
plt.plot(y_pred, label='Predicted Value')  # Plot predicted values
plt.title('LSTM Prediction vs. True Value')
plt.xlabel('Time Steps')
plt.ylabel('Value')
plt.legend()  # Add a legend to distinguish between true and predicted values
plt.show()


In [ ]:
# ====================================
# 5. Exercise: Time Series Forecasting Using a Real Dataset (Stock Prices)
# ====================================
# In this exercise, we will use a real-world stock prices dataset 
# to build an LSTM model for time series forecasting.

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import matplotlib.pyplot as plt

# Importing the yfinance package to fetch stock price data
# We will use this package to get historical stock price data from Yahoo Finance
try:
    import yfinance as yf
except ImportError:
    raise ImportError("Please install the yfinance package using: pip install yfinance")

# Download data for a specific stock (AAPL) from Yahoo Finance
# 'start' and 'end' specify the date range for historical data retrieval
stock_symbol = 'AAPL'
df_stock = yf.download(stock_symbol, start='2015-01-01', end='2023-01-01')

# Check if the data was loaded properly; handle the case where the 
# data might be empty
if df_stock.empty:
    print(f"Data for stock symbol '{stock_symbol}' could not be loaded. Please check the symbol or dates.")
else:
    # Focus on the 'Close' prices for simplicity (closing prices of the stock)
    close_prices = df_stock['Close'].values

    # Normalize the data using MinMaxScaler to scale values between 0 and 1
    # This normalization helps the LSTM model to converge more effectively
    scaler_stock = MinMaxScaler(feature_range=(0, 1))
    close_prices_scaled = scaler_stock.fit_transform(close_prices.reshape(-1, 1)).flatten()

    # Function to create a sliding window dataset for LSTM input
    # 'look_back' specifies how many previous time steps to use for prediction
    def create_dataset(data, look_back=30):
        X, Y = [], []
        for i in range(len(data) - look_back):
            X.append(data[i:(i + look_back)])  # Sequence of 'look_back' time steps
            Y.append(data[i + look_back])  # Next value to predict
        return np.array(X), np.array(Y)

    # Create input sequences and corresponding targets for the LSTM
    look_back = 30
    X_stock, y_stock = create_dataset(close_prices_scaled, look_back)
    # Reshape input data to be 3D (samples, time steps, features) as 
    # expected by LSTM
    X_stock = X_stock.reshape(X_stock.shape[0], X_stock.shape[1], 1)

    # Split the data into training and testing sets (80% train, 20% test)
    train_size_stock = int(len(X_stock) * 0.8)
    X_train_stock, X_test_stock = X_stock[:train_size_stock], X_stock[train_size_stock:]
    y_train_stock, y_test_stock = y_stock[:train_size_stock], y_stock[train_size_stock:]

    # Define an LSTM model using Keras
    model_stock = Sequential()
    # First LSTM layer with 50 units and return_sequences=True 
    # to maintain the sequential output
    model_stock.add(LSTM(50, input_shape=(look_back, 1), return_sequences=True))
    # Dropout layer for regularization (reduces overfitting)
    model_stock.add(Dropout(0.2))
    # Second LSTM layer with 50 units
    model_stock.add(LSTM(50))
    # Dense layer with a single output unit (predicting one value)
    model_stock.add(Dense(1))
    # Compile the model with Adam optimizer and mean squared error loss function
    model_stock.compile(optimizer='adam', loss='mean_squared_error')
    # Train the model on the training data, with validation on the test data
    model_stock.fit(X_train_stock, y_train_stock, epochs=20, batch_size=16, validation_data=(X_test_stock, y_test_stock), verbose=1)

    # Make predictions on the test data
    y_pred_stock = model_stock.predict(X_test_stock)
    # Inverse transform the predicted values to get original scale values
    y_pred_stock = scaler_stock.inverse_transform(y_pred_stock)
    # Inverse transform the true test values for comparison
    y_test_stock = scaler_stock.inverse_transform(y_test_stock.reshape(-1, 1))

    # Plot the true prices vs. the predicted prices
    plt.plot(y_test_stock, label='True Price')  # Plot true values
    plt.plot(y_pred_stock, label='Predicted Price')  # Plot predicted values
    plt.title('Stock Price Prediction')
    plt.xlabel('Time Steps')
    plt.ylabel('Price')
    plt.legend()
    plt.show()


In [ ]:

# Increased look_back to 60 to examine longer input sequences.
# Changed the number of neurons in LSTM layers (80 and 40 respectively).
# Added a third LSTM layer.
# Increased the dropout rate to 0.3 in one layer for further regularization.
# Adjusted optimizer parameters (learning rate of Adam).
# Increased epochs and modified batch size.

In [ ]:
# ====================================
# 5. Exercise: Time Series Forecasting Using a Real Dataset (Stock Prices) - Variation
# ====================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

# Importing the yfinance package to fetch stock price data
try:
    import yfinance as yf
except ImportError:
    raise ImportError("Please install the yfinance package using: pip install yfinance")

# Download data for a specific stock (AAPL) from Yahoo Finance
stock_symbol = 'AAPL'
df_stock = yf.download(stock_symbol, start='2015-01-01', end='2023-01-01')

if df_stock.empty:
    print(f"Data for stock symbol '{stock_symbol}' could not be loaded. Please check the symbol or dates.")
else:
    close_prices = df_stock['Close'].values

    # Normalize the data using MinMaxScaler
    scaler_stock = MinMaxScaler(feature_range=(0, 1))
    close_prices_scaled = scaler_stock.fit_transform(close_prices.reshape(-1, 1)).flatten()

    # Function to create a sliding window dataset for LSTM input
    def create_dataset(data, look_back=60):  # Increased look_back to 60
        X, Y = [], []
        for i in range(len(data) - look_back):
            X.append(data[i:(i + look_back)])
            Y.append(data[i + look_back])
        return np.array(X), np.array(Y)

    # Create input sequences and corresponding targets
    look_back = 60  # Increased from 30 to 60
    X_stock, y_stock = create_dataset(close_prices_scaled, look_back)
    X_stock = X_stock.reshape(X_stock.shape[0], X_stock.shape[1], 1)

    # Split the data into training and testing sets
    train_size_stock = int(len(X_stock) * 0.8)
    X_train_stock, X_test_stock = X_stock[:train_size_stock], X_stock[train_size_stock:]
    y_train_stock, y_test_stock = y_stock[:train_size_stock], y_stock[train_size_stock:]

    # Define an LSTM model using Keras
    model_stock = Sequential()
    # First LSTM layer with 80 units
    model_stock.add(LSTM(80, input_shape=(look_back, 1), return_sequences=True))
    # Dropout layer for regularization
    model_stock.add(Dropout(0.3))  # Increased dropout rate to 0.3
    # Second LSTM layer with 40 units
    model_stock.add(LSTM(40, return_sequences=True))
    # Third LSTM layer with 20 units
    model_stock.add(LSTM(20))
    # Dense layer with a single output unit
    model_stock.add(Dense(1))

    # Compile the model with a modified Adam optimizer
    optimizer = Adam(learning_rate=0.005)  # Custom learning rate
    model_stock.compile(optimizer=optimizer, loss='mean_squared_error')

    # Train the model with modified epochs and batch size
    model_stock.fit(X_train_stock, y_train_stock, epochs=30, batch_size=32, validation_data=(X_test_stock, y_test_stock), verbose=1)

    # Make predictions on the test data
    y_pred_stock = model_stock.predict(X_test_stock)
    y_pred_stock = scaler_stock.inverse_transform(y_pred_stock)
    y_test_stock = scaler_stock.inverse_transform(y_test_stock.reshape(-1, 1))

    # Plot the true prices vs. the predicted prices
    plt.plot(y_test_stock, label='True Price')
    plt.plot(y_pred_stock, label='Predicted Price')
    plt.title('Stock Price Prediction - Variation')
    plt.xlabel('Time Steps')
    plt.ylabel('Price')
    plt.legend()
    plt.show()


In [ ]:
# ====================================
# 6. Exercise Variation: Time Series Forecasting Using a Real Dataset (Temperature)
# ====================================
# In this exercise, we will use a real-world weather dataset to build an LSTM model for time series forecasting.

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import matplotlib.pyplot as plt

# Load a weather dataset
# We will use a publicly available weather dataset (daily temperatures in a specific location)
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv'
df_weather = pd.read_csv(url, parse_dates=['Date'], index_col='Date')

# Display the first few rows
print(df_weather.head())

# Focus on the 'Temp' column (daily minimum temperature)
temperatures = df_weather['Temp'].values

# Preprocessing: Normalize and create a sliding window dataset
scaler_temp = MinMaxScaler(feature_range=(0, 1))
temperatures_scaled = scaler_temp.fit_transform(temperatures.reshape(-1, 1)).flatten()

# Function to create dataset for LSTM
def create_dataset(data, look_back=30):
    X, Y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i:(i + look_back)])
        Y.append(data[i + look_back])
    return np.array(X), np.array(Y)

# Create sequences
look_back = 30
X_temp, y_temp = create_dataset(temperatures_scaled, look_back)
X_temp = X_temp.reshape(X_temp.shape[0], X_temp.shape[1], 1)

# Split data into train and test sets
train_size_temp = int(len(X_temp) * 0.8)
X_train_temp, X_test_temp = X_temp[:train_size_temp], X_temp[train_size_temp:]
y_train_temp, y_test_temp = y_temp[:train_size_temp], y_temp[train_size_temp:]

# Define and train the LSTM model
model_temp = Sequential()
model_temp.add(LSTM(50, input_shape=(look_back, 1), return_sequences=True))
model_temp.add(Dropout(0.2))
model_temp.add(LSTM(50))
model_temp.add(Dense(1))
model_temp.compile(optimizer='adam', loss='mean_squared_error')
model_temp.fit(X_train_temp, y_train_temp, epochs=20, batch_size=16, validation_data=(X_test_temp, y_test_temp), verbose=1)

# Predict and plot results
y_pred_temp = model_temp.predict(X_test_temp)
y_pred_temp = scaler_temp.inverse_transform(y_pred_temp)
y_test_temp = scaler_temp.inverse_transform(y_test_temp.reshape(-1, 1))

plt.plot(y_test_temp, label='True Temperature')
plt.plot(y_pred_temp, label='Predicted Temperature')
plt.title('Temperature Prediction')
plt.xlabel('Time Steps')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.show()
